In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# clean corrupted images
def clean(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            path = os.path.join(root, file)
            try:
                img = Image.open(path)
                img.verify()
            except:
                print("Removing bad file:", path)
                os.remove(path)

clean("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained")
clean("/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed")
clean( "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested")

In [ ]:

train_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/trained"
val_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/valed"
test_path = "/content/drive/MyDrive/1. All Semester study/1. All about Machine Learning/2. Farhana/Datasets/chest_xray/tested"

In [ ]:
#Setup Data Generators with Augmentation

img_size = 224
batch_size = 32

datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

datagen_val_test = ImageDataGenerator(rescale=1./255)

train_generator = datagen_train.flow_from_directory(
    train_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = datagen_val_test.flow_from_directory(
    val_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = datagen_val_test.flow_from_directory(
    test_path,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Found 5215 images belonging to 3 classes.
Found 300 images belonging to 3 classes.
Found 624 images belonging to 3 classes.


In [ ]:
# Build Model with EfficientNetB0

base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=(img_size, img_size, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(3, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Compile the Model
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Setup Early Stopping

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)


In [ ]:
for class_name, idx in train_generator.class_indices.items():
    print(f"{class_name}: {np.sum(train_generator.labels == idx)} images")

bacterial: 2529 images
normal: 1341 images
viral: 1345 images


In [ ]:
# Get weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

# Convert to dictionary
class_weights = dict(enumerate(class_weights))
print("✅ Computed class weights:", class_weights)

✅ Computed class weights: {0: np.float64(0.6873599578225913), 1: np.float64(1.2962962962962963), 2: np.float64(1.292441140024783)}


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    class_weight=class_weights,
    callbacks=[early_stop]
)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 506s 3s/step - accuracy: 0.3486 - loss: 1.1228 - val_accuracy: 0.3333 - val_loss: 1.1038
Epoch 2/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 510s 3s/step - accuracy: 0.3393 - loss: 1.1122 - val_accuracy: 0.3333 - val_loss: 1.0993
Epoch 3/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 528s 3s/step - accuracy: 0.3263 - loss: 1.1057 - val_accuracy: 0.3333 - val_loss: 1.0988
Epoch 4/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 486s 3s/step - accuracy: 0.3370 - loss: 1.1128 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 5/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 492s 3s/step - accuracy: 0.3178 - loss: 1.1049 - val_accuracy: 0.3333 - val_loss: 1.0988
Epoch 6/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 479s 3s/step - accuracy: 0.3075 - loss: 1.1090 - val_accuracy: 0.3333 - val_loss: 1.0988
Epoch 7/30
163/163 ━━━━━━━━━━━━━━━━━━━━ 478s 3s/step - accuracy: 0.3283 - loss: 1.1012 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 8/30
 32/163 ━━━━━━━━━━━━━━━━━━━━ 5:51 3s/step - accuracy: 0.3813 - loss: 1.1160

In [ ]:
print("Train classes:", train_generator.class_indices)
print("Val classes:", val_generator.class_indices)
print("Train samples:", train_generator.samples)
print("Val samples:", val_generator.samples)
